# AcidNet Qwen3.5-4B LoRA Colab Path

This notebook restores the published AcidNet runtime-dialogue dataset from Hugging Face, fine-tunes a LoRA adapter with Unsloth, and can optionally export an adapter GGUF plus upload the fresh run back to the Hub.

Important:
- Training uses the Hugging Face base model (`Qwen/Qwen3.5-4B`), not the deployment GGUF.
- `Qwen3.5-4B-Q4_K_M.gguf` is a deployment artifact for `llama-server`, not the primary fine-tuning input.
- The published AcidNet dataset repo uses stable paths under `runs/<run-name>/...`.
- In Colab, prefer a `Secrets` entry named `HF_TOKEN`; the notebook falls back to the normal `HF_TOKEN` environment variable only if the secret is absent.
- The published run spec is restored and used as the default hyperparameter source when available.
- `TRAIN_MODE=auto` prefers the full dataset only on GPUs that support `bf16`; on T4-class or other non-`bf16` GPUs it warns and switches to the smoke split by default.


In [ ]:
%%capture
import sys
import subprocess

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-U",
        "unsloth",
        "datasets",
        "transformers",
        "trl",
        "accelerate",
        "peft",
        "bitsandbytes",
        "huggingface_hub",
        "sentencepiece",
        "protobuf",
    ],
    check=True,
)


In [ ]:
from __future__ import annotations

from datetime import datetime, timezone
from pathlib import Path
import os

WORKDIR = Path("/content/acidnet")
DATA_ROOT = WORKDIR / "data"
PROMPT_PACK_DIR = DATA_ROOT / "prompt_packs"
SFT_DIR = DATA_ROOT / "sft"
PREFERENCES_DIR = DATA_ROOT / "preferences"
TRAINING_DIR = DATA_ROOT / "training"
EVAL_DIR = DATA_ROOT / "eval"
GGUF_DIR = DATA_ROOT / "gguf"
MODELS_DIR = WORKDIR / "models"

for path in [
    WORKDIR,
    PROMPT_PACK_DIR,
    SFT_DIR,
    PREFERENCES_DIR,
    TRAINING_DIR,
    EVAL_DIR,
    GGUF_DIR,
    MODELS_DIR,
]:
    path.mkdir(parents=True, exist_ok=True)

def resolve_hf_token() -> tuple[str, str]:
    try:
        from google.colab import userdata  # type: ignore
        secret_token = userdata.get("HF_TOKEN")
        if secret_token:
            return secret_token, "colab_secret"
    except Exception:
        pass
    env_token = os.environ.get("HF_TOKEN", "")
    if env_token:
        return env_token, "environment"
    return "", "missing"

HF_TOKEN, HF_TOKEN_SOURCE = resolve_hf_token()
DATASET_REPO = os.environ.get("ACIDNET_DATASET_REPO", "acidsound/acidnet_dataset")
MODEL_REPO = os.environ.get("ACIDNET_MODEL_REPO", "acidsound/acidnet_model")
DATASET_RUN = os.environ.get(
    "ACIDNET_DATASET_RUN",
    "qwen3_5_4b_runtime_dialogue_unsloth_wsl_full_headsync_20260308",
)
OUTPUT_RUN = os.environ.get(
    "ACIDNET_OUTPUT_RUN",
    f"qwen3_5_4b_runtime_dialogue_colab_{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')}",
)
BASE_MODEL = os.environ.get("ACIDNET_BASE_MODEL", "Qwen/Qwen3.5-4B")
BASE_GGUF_URL = os.environ.get(
    "ACIDNET_BASE_GGUF_URL",
    "https://huggingface.co/unsloth/Qwen3.5-4B-GGUF/resolve/main/Qwen3.5-4B-Q4_K_M.gguf",
)
BASE_GGUF_PATH = MODELS_DIR / "Qwen3.5-4B-Q4_K_M.gguf"

TRAIN_MODE = os.environ.get("ACIDNET_TRAIN_MODE", "auto").strip().lower()
ALLOW_SLOW_FULL = os.environ.get("ACIDNET_ALLOW_SLOW_FULL", "false").strip().lower() in {"1", "true", "yes", "on"}
LOAD_IN_4BIT = True
MAX_SEQ_LENGTH = 1024
LEARNING_RATE = 2e-4
PER_DEVICE_TRAIN_BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 8
NUM_TRAIN_EPOCHS = 1
WARMUP_RATIO = 0.05
EVAL_STEPS = 1000
SAVE_STEPS = 1000
LORA_RANK = 16
LORA_ALPHA = 16

UPLOAD_ADAPTER_TO_HF = False
EXPORT_ADAPTER_GGUF = False
UPLOAD_GGUF_TO_HF = False
DOWNLOAD_BASE_GGUF = False

ADAPTER_DIR = TRAINING_DIR / f"{OUTPUT_RUN}_adapter"
LOCAL_RUN_SPEC_PATH = TRAINING_DIR / f"{OUTPUT_RUN}_run_spec.json"
LOCAL_COLAB_MANIFEST_PATH = TRAINING_DIR / f"{OUTPUT_RUN}_colab_manifest.json"
ADAPTER_GGUF_PATH = GGUF_DIR / f"{OUTPUT_RUN}_adapter-f16.gguf"

print({
    "hf_token_source": HF_TOKEN_SOURCE,
    "dataset_repo": DATASET_REPO,
    "model_repo": MODEL_REPO,
    "dataset_run": DATASET_RUN,
    "output_run": OUTPUT_RUN,
    "train_mode": TRAIN_MODE,
    "allow_slow_full": ALLOW_SLOW_FULL,
    "base_model": BASE_MODEL,
    "load_in_4bit": LOAD_IN_4BIT,
})


In [ ]:
from huggingface_hub import login, notebook_login

if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=False)
    print(f"Logged into Hugging Face with HF_TOKEN from {HF_TOKEN_SOURCE}.")
else:
    print("HF_TOKEN is empty. Add a Colab secret named HF_TOKEN or run notebook_login() below.")
    # notebook_login()


In [ ]:
from huggingface_hub import hf_hub_download
import shutil

required_dataset_files = {
    f"runs/{DATASET_RUN}/sft/train.jsonl": SFT_DIR / "train.jsonl",
    f"runs/{DATASET_RUN}/sft/eval.jsonl": SFT_DIR / "eval.jsonl",
}

optional_dataset_files = {
    f"runs/{DATASET_RUN}/prompt_packs/requests.parquet": PROMPT_PACK_DIR / "requests.parquet",
    f"runs/{DATASET_RUN}/prompt_packs/teacher_outputs.jsonl": PROMPT_PACK_DIR / "teacher_outputs.jsonl",
    f"runs/{DATASET_RUN}/sft/bench_train_1024.jsonl": SFT_DIR / "bench_train_1024.jsonl",
    f"runs/{DATASET_RUN}/sft/bench_eval_128.jsonl": SFT_DIR / "bench_eval_128.jsonl",
    f"runs/{DATASET_RUN}/preferences/preferences.parquet": PREFERENCES_DIR / "preferences.parquet",
    f"runs/{DATASET_RUN}/preferences/manifest.json": PREFERENCES_DIR / "manifest.json",
    f"runs/{DATASET_RUN}/manifests/pipeline.json": TRAINING_DIR / "published_pipeline.json",
    f"runs/{DATASET_RUN}/manifests/run_spec.json": TRAINING_DIR / "published_run_spec.json",
    f"runs/{DATASET_RUN}/manifests/gate_report.json": EVAL_DIR / "published_gate_report.json",
    f"runs/{DATASET_RUN}/manifests/publish_manifest.json": TRAINING_DIR / "published_hf_manifest.json",
}

def restore_hf_file(remote_path: str, local_path: Path, *, required: bool) -> Path | None:
    try:
        cached = hf_hub_download(
            repo_id=DATASET_REPO,
            repo_type="dataset",
            filename=remote_path,
            token=HF_TOKEN or None,
        )
    except Exception as exc:
        if required:
            raise
        print(f"skip optional {remote_path}: {exc}")
        return None
    local_path.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(cached, local_path)
    print(f"restored {remote_path} -> {local_path}")
    return local_path

for remote_path, local_path in required_dataset_files.items():
    restore_hf_file(remote_path, local_path, required=True)

for remote_path, local_path in optional_dataset_files.items():
    restore_hf_file(remote_path, local_path, required=False)


In [ ]:
import json

published_run_spec_path = TRAINING_DIR / "published_run_spec.json"
if published_run_spec_path.exists():
    published_spec = json.loads(published_run_spec_path.read_text(encoding="utf-8"))
    BASE_MODEL = published_spec.get("model_name", BASE_MODEL)
    MAX_SEQ_LENGTH = int(published_spec.get("max_seq_length", MAX_SEQ_LENGTH))
    LEARNING_RATE = float(published_spec.get("learning_rate", LEARNING_RATE))
    PER_DEVICE_TRAIN_BATCH_SIZE = int(
        published_spec.get("per_device_train_batch_size", PER_DEVICE_TRAIN_BATCH_SIZE)
    )
    GRADIENT_ACCUMULATION_STEPS = int(
        published_spec.get("gradient_accumulation_steps", GRADIENT_ACCUMULATION_STEPS)
    )
    NUM_TRAIN_EPOCHS = int(published_spec.get("num_train_epochs", NUM_TRAIN_EPOCHS))
    WARMUP_RATIO = float(published_spec.get("warmup_ratio", WARMUP_RATIO))
    EVAL_STEPS = int(published_spec.get("eval_steps", EVAL_STEPS))
    SAVE_STEPS = int(published_spec.get("save_steps", SAVE_STEPS))
    LORA_RANK = int(published_spec.get("lora_rank", LORA_RANK))
    LORA_ALPHA = int(published_spec.get("lora_alpha", LORA_ALPHA))

print({
    "base_model": BASE_MODEL,
    "max_seq_length": MAX_SEQ_LENGTH,
    "learning_rate": LEARNING_RATE,
    "batch_size": PER_DEVICE_TRAIN_BATCH_SIZE,
    "grad_accum": GRADIENT_ACCUMULATION_STEPS,
    "epochs": NUM_TRAIN_EPOCHS,
    "lora_rank": LORA_RANK,
    "lora_alpha": LORA_ALPHA,
})


In [ ]:
from datasets import load_dataset
import json
import math
import torch

def _bool_env(value: str) -> bool:
    return value.strip().lower() in {"1", "true", "yes", "on"}

def ensure_smoke_jsonl(full_path: Path, smoke_path: Path, limit: int) -> Path:
    if smoke_path.exists():
        return smoke_path
    full_dataset = load_dataset("json", data_files=str(full_path), split="train")
    rows = [full_dataset[index] for index in range(min(limit, len(full_dataset)))]
    smoke_path.parent.mkdir(parents=True, exist_ok=True)
    with smoke_path.open("w", encoding="utf-8") as handle:
        for row in rows:
            handle.write(json.dumps(row, ensure_ascii=False) + "\n")
    return smoke_path

gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"
bf16_supported = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
gpu_upper = gpu_name.upper()
slow_gpu_markers = ("T4", "V100", "P100", "P4", "K80")
looks_slow_for_full = any(marker in gpu_upper for marker in slow_gpu_markers)

requested_train_mode = TRAIN_MODE
if requested_train_mode not in {"auto", "smoke", "full"}:
    raise ValueError("ACIDNET_TRAIN_MODE must be one of auto, smoke, full.")

if requested_train_mode == "auto":
    if (not torch.cuda.is_available()) or (not bf16_supported) or looks_slow_for_full:
        SELECTED_TRAIN_MODE = "smoke"
        SELECTED_TRAIN_REASON = "auto_smoke_due_to_gpu"
    else:
        SELECTED_TRAIN_MODE = "full"
        SELECTED_TRAIN_REASON = "auto_full"
elif requested_train_mode == "full" and (not bf16_supported) and (not ALLOW_SLOW_FULL):
    SELECTED_TRAIN_MODE = "smoke"
    SELECTED_TRAIN_REASON = "forced_smoke_due_to_no_bf16"
else:
    SELECTED_TRAIN_MODE = requested_train_mode
    SELECTED_TRAIN_REASON = "manual_override"

FULL_TRAIN_PATH = SFT_DIR / "train.jsonl"
FULL_EVAL_PATH = SFT_DIR / "eval.jsonl"
SMOKE_TRAIN_PATH = ensure_smoke_jsonl(FULL_TRAIN_PATH, SFT_DIR / "bench_train_1024.jsonl", 1024)
SMOKE_EVAL_PATH = ensure_smoke_jsonl(FULL_EVAL_PATH, SFT_DIR / "bench_eval_128.jsonl", 128)

if SELECTED_TRAIN_MODE == "full":
    SELECTED_TRAIN_PATH = FULL_TRAIN_PATH
    SELECTED_EVAL_PATH = FULL_EVAL_PATH
else:
    SELECTED_TRAIN_PATH = SMOKE_TRAIN_PATH
    SELECTED_EVAL_PATH = SMOKE_EVAL_PATH

train_raw = load_dataset("json", data_files=str(SELECTED_TRAIN_PATH), split="train")
eval_raw = load_dataset("json", data_files=str(SELECTED_EVAL_PATH), split="train")

effective_batch_size = PER_DEVICE_TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS
steps_per_epoch = max(1, math.ceil(len(train_raw) / effective_batch_size))
total_steps = max(1, steps_per_epoch * NUM_TRAIN_EPOCHS)
WARMUP_STEPS = max(1, int(total_steps * WARMUP_RATIO))

if SELECTED_TRAIN_MODE == "smoke":
    print("Warning: slow or non-bf16 GPU detected. Using smoke dataset by default.")

print({
    "gpu_name": gpu_name,
    "bf16_supported": bf16_supported,
    "requested_train_mode": requested_train_mode,
    "selected_train_mode": SELECTED_TRAIN_MODE,
    "selected_train_reason": SELECTED_TRAIN_REASON,
    "selected_train_path": str(SELECTED_TRAIN_PATH),
    "selected_eval_path": str(SELECTED_EVAL_PATH),
    "train_rows": len(train_raw),
    "eval_rows": len(eval_raw),
    "effective_batch_size": effective_batch_size,
    "steps_per_epoch": steps_per_epoch,
    "estimated_total_steps": total_steps,
    "warmup_steps": WARMUP_STEPS,
})

print(train_raw[0]["custom_id"])
train_raw[0]


In [ ]:
from unsloth import FastLanguageModel
from trl import SFTConfig, SFTTrainer
import json

def format_record(record, tokenizer):
    if record.get("text"):
        return {"text": record["text"]}
    messages = record["messages"]
    if isinstance(messages, str):
        messages = json.loads(messages)
    if hasattr(tokenizer, "apply_chat_template"):
        try:
            text = tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=False,
                enable_thinking=False,
            )
            return {"text": text}
        except TypeError:
            text = tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=False,
            )
            return {"text": text}
    return {"text": "\n".join(f"{item['role']}: {item['content']}" for item in messages)}

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=LOAD_IN_4BIT,
    full_finetuning=False,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    use_gradient_checkpointing="unsloth",
)

train_dataset = train_raw.map(
    lambda record: format_record(record, tokenizer),
    remove_columns=train_raw.column_names,
)
eval_dataset = eval_raw.map(
    lambda record: format_record(record, tokenizer),
    remove_columns=eval_raw.column_names,
)

fp16_enabled = torch.cuda.is_available() and not bf16_supported

print({
    "selected_train_mode": SELECTED_TRAIN_MODE,
    "selected_train_path": str(SELECTED_TRAIN_PATH),
    "selected_eval_path": str(SELECTED_EVAL_PATH),
    "bf16": bf16_supported,
    "fp16": fp16_enabled,
    "train_text_preview": train_dataset[0]["text"][:400],
})


In [ ]:
base_tokenizer = getattr(tokenizer, "tokenizer", tokenizer)

if base_tokenizer.pad_token is None:
    base_tokenizer.pad_token = base_tokenizer.eos_token

def tokenize_batch(batch):
    return base_tokenizer(
        text=batch["text"],
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
    )

tokenized_train_dataset = train_dataset.map(
    tokenize_batch,
    batched=True,
    remove_columns=train_dataset.column_names,
)
tokenized_eval_dataset = eval_dataset.map(
    tokenize_batch,
    batched=True,
    remove_columns=eval_dataset.column_names,
)

print({
    "base_tokenizer_class": type(base_tokenizer).__name__,
    "tokenized_example": tokenized_train_dataset[0],
})


In [ ]:
FastLanguageModel.for_training(model)

from transformers import DataCollatorForLanguageModeling, Trainer, TrainingArguments

data_collator = DataCollatorForLanguageModeling(tokenizer=base_tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    processing_class=base_tokenizer,
    data_collator=data_collator,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_eval_dataset,
    args=TrainingArguments(
        output_dir=str(ADAPTER_DIR),
        learning_rate=LEARNING_RATE,
        per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        num_train_epochs=NUM_TRAIN_EPOCHS,
        warmup_steps=WARMUP_STEPS,
        eval_steps=EVAL_STEPS,
        save_steps=SAVE_STEPS,
        bf16=bf16_supported,
        fp16=fp16_enabled,
        optim="adamw_8bit",
        logging_steps=10,
        eval_strategy="steps",
        save_strategy="steps",
        remove_unused_columns=False,
        seed=3407,
        report_to="none",
    ),
)

train_result = trainer.train()
print(train_result.metrics)


In [ ]:
import platform

trainer.save_model(str(ADAPTER_DIR))
base_tokenizer.save_pretrained(str(ADAPTER_DIR))

local_run_spec = {
    "experiment_key": "qwen3_5_4b_baseline",
    "model_name": BASE_MODEL,
    "output_dir": str(ADAPTER_DIR),
    "dataset_text_field": "text",
    "requested_train_mode": TRAIN_MODE,
    "selected_train_mode": SELECTED_TRAIN_MODE,
    "selected_train_reason": SELECTED_TRAIN_REASON,
    "max_seq_length": MAX_SEQ_LENGTH,
    "learning_rate": LEARNING_RATE,
    "per_device_train_batch_size": PER_DEVICE_TRAIN_BATCH_SIZE,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "num_train_epochs": NUM_TRAIN_EPOCHS,
    "warmup_ratio": WARMUP_RATIO,
    "warmup_steps": WARMUP_STEPS,
    "eval_steps": EVAL_STEPS,
    "save_steps": SAVE_STEPS,
    "lora_rank": LORA_RANK,
    "lora_alpha": LORA_ALPHA,
    "bf16": bf16_supported,
    "train_dataset_path": str(SELECTED_TRAIN_PATH),
    "eval_dataset_path": str(SELECTED_EVAL_PATH),
}
LOCAL_RUN_SPEC_PATH.write_text(json.dumps(local_run_spec, indent=2), encoding="utf-8")

colab_manifest = {
    "output_run": OUTPUT_RUN,
    "dataset_repo": DATASET_REPO,
    "dataset_run": DATASET_RUN,
    "model_repo": MODEL_REPO,
    "requested_train_mode": TRAIN_MODE,
    "selected_train_mode": SELECTED_TRAIN_MODE,
    "selected_train_reason": SELECTED_TRAIN_REASON,
    "selected_train_path": str(SELECTED_TRAIN_PATH),
    "selected_eval_path": str(SELECTED_EVAL_PATH),
    "adapter_dir": str(ADAPTER_DIR),
    "run_spec": str(LOCAL_RUN_SPEC_PATH),
    "base_model": BASE_MODEL,
    "load_in_4bit": LOAD_IN_4BIT,
    "gpu_name": gpu_name,
    "bf16_supported": bf16_supported,
    "platform": platform.platform(),
}
LOCAL_COLAB_MANIFEST_PATH.write_text(json.dumps(colab_manifest, indent=2), encoding="utf-8")

print({
    "adapter_dir": str(ADAPTER_DIR),
    "run_spec": str(LOCAL_RUN_SPEC_PATH),
    "manifest": str(LOCAL_COLAB_MANIFEST_PATH),
})


In [ ]:
from huggingface_hub import HfApi

if UPLOAD_ADAPTER_TO_HF:
    api = HfApi(token=HF_TOKEN or None)
    api.create_repo(repo_id=MODEL_REPO, repo_type="model", private=True, exist_ok=True)
    api.upload_folder(
        folder_path=str(ADAPTER_DIR),
        path_in_repo=f"runs/{OUTPUT_RUN}/adapter",
        repo_id=MODEL_REPO,
        repo_type="model",
        commit_message=f"Add adapter for {OUTPUT_RUN}",
    )
    api.upload_file(
        path_or_fileobj=str(LOCAL_RUN_SPEC_PATH),
        path_in_repo=f"runs/{OUTPUT_RUN}/manifests/run_spec.json",
        repo_id=MODEL_REPO,
        repo_type="model",
        commit_message=f"Add run spec for {OUTPUT_RUN}",
    )
    api.upload_file(
        path_or_fileobj=str(LOCAL_COLAB_MANIFEST_PATH),
        path_in_repo=f"runs/{OUTPUT_RUN}/manifests/colab_manifest.json",
        repo_id=MODEL_REPO,
        repo_type="model",
        commit_message=f"Add Colab manifest for {OUTPUT_RUN}",
    )
    print(f"Uploaded adapter run to {MODEL_REPO}.")
else:
    print("UPLOAD_ADAPTER_TO_HF is False. Skipping adapter upload.")


In [ ]:
import subprocess
import sys
import urllib.request
from huggingface_hub import HfApi

LLAMA_CPP_DIR = WORKDIR / "vendor" / "llama.cpp"

if DOWNLOAD_BASE_GGUF and not BASE_GGUF_PATH.exists():
    urllib.request.urlretrieve(BASE_GGUF_URL, BASE_GGUF_PATH)
    print(f"Downloaded base GGUF -> {BASE_GGUF_PATH}")

if EXPORT_ADAPTER_GGUF:
    if not LLAMA_CPP_DIR.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/ggml-org/llama.cpp.git", str(LLAMA_CPP_DIR)],
            check=True,
        )
    subprocess.run(
        [
            sys.executable,
            str(LLAMA_CPP_DIR / "convert_lora_to_gguf.py"),
            str(ADAPTER_DIR),
            "--outfile",
            str(ADAPTER_GGUF_PATH),
            "--outtype",
            "f16",
            "--base-model-id",
            BASE_MODEL,
        ],
        check=True,
    )
    print(f"Exported adapter GGUF -> {ADAPTER_GGUF_PATH}")
    if UPLOAD_GGUF_TO_HF:
        api = HfApi(token=HF_TOKEN or None)
        api.create_repo(repo_id=MODEL_REPO, repo_type="model", private=True, exist_ok=True)
        api.upload_file(
            path_or_fileobj=str(ADAPTER_GGUF_PATH),
            path_in_repo=f"runs/{OUTPUT_RUN}/gguf/adapter-f16.gguf",
            repo_id=MODEL_REPO,
            repo_type="model",
            commit_message=f"Add adapter GGUF for {OUTPUT_RUN}",
        )
        print(f"Uploaded adapter GGUF to {MODEL_REPO}.")
else:
    print("EXPORT_ADAPTER_GGUF is False. Skipping GGUF export.")


## Notes

- The fresh adapter lives under `data/training/<output-run>_adapter` in this Colab workspace.
- In `TRAIN_MODE=auto`, non-`bf16` GPUs such as T4 will train against the smoke split unless `ACIDNET_ALLOW_SLOW_FULL=true` is set explicitly.
- The restored train/eval dataset lives under `data/sft/train.jsonl` and `data/sft/eval.jsonl`.
- The restored prompt-pack provenance, preference bundle, and manifests stay under the same repo-like `data/...` layout as the local project.
- If you want a runtime deployable artifact, export `adapter-f16.gguf` and then combine it with the base `Qwen3.5-4B-Q4_K_M.gguf` on the serving machine.
- For local AcidNet runtime, the promoted base GGUF restore path is `models/Qwen3.5-4B-Q4_K_M.gguf`.
